# 분석 단계별 프로세스 - 랜덤 포레스트 회귀 + LP 인력배치 최적화

1. 라이브러리 / 데이터 불러오기
2. 데이터 종류 및 개수 확인
3. 데이터 정제(전처리)
4. 랜덤포레스트 모델학습 및 결과분석
5. 선형계획(LP) 기반 인력배치 최적화 (가이드북 재현 + 확장)

목표: 가이드북이 보고한 train MSE 185.98 / test MSE 183.06 과 같은 자릿수의 결과가 재현되는지 확인하고, 이후 XGBoost/LightGBM과 비교할 기준점을 만든다.

> 가이드북은 정확한 피처/타겟/분할 방식을 코드로 전부 공개하지 않았다. 여기서는 15분/30분/45분/60분 서브컬럼(평균과 상관계수 0.99+, 사실상 평균의 구성요소)을 피처에서 제외해 데이터 누수를 막고,
> 시간 순서를 지키기 위해 RNN 재현과 동일하게 9월 1~14일(336시간)을 테스트셋으로 고정했다.

## 1. 라이브러리 / 데이터 불러오기

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

DATA_PATH = Path("../data/okm_augumented_2021.csv")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)  # 결과물을 results/ 폴더에 정리 저장

df = pd.read_csv(DATA_PATH)

## 2. 데이터 종류 및 개수 확인

In [2]:
print(f"[정보] 데이터 shape: {df.shape}")
df.info()
df.head()

[정보] 데이터 shape: (6168, 18)
<class 'pandas.DataFrame'>
RangeIndex: 6168 entries, 0 to 6167
Data columns (total 18 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   날짜        6168 non-null   int64  
 1   시간        6168 non-null   int64  
 2   15분       6168 non-null   int64  
 3   30분       6168 non-null   int64  
 4   45분       6168 non-null   int64  
 5   60분       6168 non-null   int64  
 6   평균        6168 non-null   int64  
 7   생산량       6168 non-null   int64  
 8   기온        6168 non-null   float64
 9   풍속        6165 non-null   float64
 10  습도        6168 non-null   int64  
 11  강수량       6167 non-null   float64
 12  전기요금(계절)  6168 non-null   float64
 13  day       6168 non-null   int64  
 14  d         6168 non-null   int64  
 15  m         6168 non-null   int64  
 16  공장인원      6151 non-null   float64
 17  인건비       6168 non-null   float64
dtypes: float64(6), int64(12)
memory usage: 867.5 KB


,날짜,시간,15분,30분,45분,60분,평균,생산량,기온,풍속,습도,강수량,전기요금(계절),day,d,m,공장인원,인건비
0,20210101,0,62,61,61,61,61,0,-3.2,2.4,71,0.0,109.8,5,1,1,0.0,1.5
1,20210101,1,96,93,116,113,105,0,-4.5,1.5,77,0.0,109.8,5,1,1,0.0,1.5
2,20210101,2,106,96,106,107,104,0,-3.9,2.6,58,0.0,109.8,5,1,1,0.0,1.5
3,20210101,3,92,110,110,109,105,0,-4.1,2.6,56,0.0,109.8,5,1,1,0.0,1.5
4,20210101,4,108,105,106,108,107,0,-4.6,2.6,60,0.0,109.8,5,1,1,0.0,1.5


## 3. 데이터 정제(전처리)

### 3-1. 결측치 처리
풍속(3)·강수량(1)은 시계열 보간, 공장인원(17)은 전부 생산량=0 구간이므로 0으로 채운다 (01의 데이터 품질진단 결과에 근거).

In [3]:
df["풍속"] = df["풍속"].interpolate()
df["강수량"] = df["강수량"].interpolate().fillna(0)
df["공장인원"] = df["공장인원"].fillna(0)
df.isna().sum().sum()

np.int64(0)

### 3-2. 이상치 보정: "시간" 컬럼 (2021-07-13, 07-15)

0~23 범위를 벗어난 48개 행. 두 날짜 모두 24행이 그대로 존재하고 앞뒤 날짜와 행 순서도 이어져 있어, 데이터가 밀리거나 빠진 게 아니라 `시간` 값 자체만 깨진 것으로 판단한다.
두 날짜 모두 생산량이 24시간 내내 0이라 다툴 여지가 없으므로, 각 날짜 안에서 행이 등장한 순서를 그대로 0~23시로 재구성한다(행 삭제 없음).

In [4]:
# 두 날짜 모두 24행이 그대로 있고 앞뒤 날짜와 행 순서가 이어져 있어(진단 완료),
# 데이터 누락이 아니라 "시간" 값만 깨진 것으로 판단. 두 날짜 모두 생산량이
# 24시간 내내 0이라 순서 재구성에 무리가 없어, 날짜별 등장 순서를 0~23시로
# 다시 채운다(행 삭제 없이 보정).
anomaly_dates = df.loc[~df["시간"].between(0, 23), "날짜"].unique()
for d in anomaly_dates:
    idx = df.index[df["날짜"] == d]
    df.loc[idx, "시간"] = range(len(idx))

remaining_violations = (~df["시간"].between(0, 23)).sum()
print(f"[정보] '시간' 이상치 보정 완료: 대상 날짜 {list(anomaly_dates)} -> 보정 후 범위 위반 {remaining_violations}건")

[정보] '시간' 이상치 보정 완료: 대상 날짜 [np.int64(20210713), np.int64(20210715)] -> 보정 후 범위 위반 0건


### 3-3. 타깃 / 피처 선정

타깃은 `평균`. 평균의 구성요소인 15분/30분/45분/60분과 `날짜`는 누수 방지를 위해 피처에서 제외한다.

In [5]:
TARGET = "평균"
LEAK_COLS = ["15분", "30분", "45분", "60분", TARGET, "날짜"]
feature_cols = [c for c in df.columns if c not in LEAK_COLS]
print(f"[정보] 사용 피처({len(feature_cols)}개): {feature_cols}")

X = df[feature_cols]
y = df[TARGET]

[정보] 사용 피처(12개): ['시간', '생산량', '기온', '풍속', '습도', '강수량', '전기요금(계절)', 'day', 'd', 'm', '공장인원', '인건비']


### 3-4. 시간순 분할

9/1 ~ 9/14(336시간)을 테스트로, 나머지를 학습으로 사용

In [6]:
test_mask = df["날짜"] >= 20210901
X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]
print(f"[정보] train={len(X_train)}행 / test={len(X_test)}행")

[정보] train=5832행 / test=336행


## 4. 랜덤포레스트 모델학습 및 결과분석

In [7]:
forest_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
forest_reg.fit(X_train, y_train)

train_mse = mean_squared_error(y_train, forest_reg.predict(X_train))
test_mse = mean_squared_error(y_test, forest_reg.predict(X_test))

print("=== 결과 ===")
print(f"train MSE: {train_mse:.4f}   (가이드북 참고값: 185.98)")
print(f"test  MSE: {test_mse:.4f}   (가이드북 참고값: 183.06)")

with open(RESULTS_DIR / "rf_metrics.txt", "w") as f:
    f.write(f"train_mse={train_mse:.4f}\ntest_mse={test_mse:.4f}\n")

=== 결과 ===
train MSE: 23.2458   (가이드북 참고값: 185.98)
test  MSE: 130.8202   (가이드북 참고값: 183.06)


### 4-1. 변수중요도

In [8]:
importances = pd.Series(forest_reg.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.to_csv(RESULTS_DIR / "rf_feature_importance.csv")
importances.to_frame("importance")

,importance
생산량,0.379414
공장인원,0.253283
시간,0.114835
day,0.077864
m,0.053993
d,0.045626
습도,0.025424
기온,0.025228
풍속,0.010015
전기요금(계절),0.008507


## 5. 선형계획(LP) 기반 인력배치 최적화 (가이드북 재현 + 확장)

가이드북 공개 내용:

- 목적함수 `Cost = P_human + P_electric` (최소화)
- 제약조건
  - `1 <= P_electric <= 2`
  - `1 <= P_human <= 75`
  - `11 <= 2*P_electric + P_human <= 95`
- 공개된 예시 출력: "생산량이 0.17183770899999998일 때, 낮에 공장직원 5.0을 투입시켜야한다", 최종 해 610.4238440032458

> **투명성 고지**: 가이드북은 "생산량"이 이 LP에 정확히 어떻게 연결되는지(목적함수 계수인지, 제약조건의 우변인지)와 610.42라는 최종값이 단일 해인지 여러 시점을 합산한 값인지를 코드로 공개하지 않았다.
> 그래서 100% 동일 재현은 불가능하고, 두 단계로 나눠 재현/확장한다.
>
> - **Part A**: 가이드북에 문자 그대로 공개된 제약조건만으로 "생산량 무관 정적 LP"를 1회 풀어, 공개된 구조 자체가 어떤 해를 내는지 확인한다.
> - **Part B**: 가이드북에는 없는 **우리의 확장**. 목적함수의 계수를 실제 데이터의 시간당 인건비/전기요금(계절)으로 바꿔 전체 데이터셋의 모든 시간(행)에 대해 LP를 반복 실행 → 시간대별 최적 인원배치를 구하고 합산한다. 가이드북 수치(610.42, 5.0명)와의 일치를 주장하지 않는 참고용 확장이다.

구현: `scipy.optimize.linprog`(HiGHS)를 사용한다. 외부 실행파일(CBC 등)을 부르지 않고 파이썬 프로세스 안에서 계산하므로 행마다 반복 실행해도 빠르다.

In [9]:
from scipy.optimize import linprog

# 공통 제약조건: 1<=P_electric<=2, 1<=P_human<=75, 11<=2*P_electric+P_human<=95
# linprog는 "<=" 부등식만 받으므로 2*Pe+Ph>=11 은 -2*Pe-Ph<=-11 로 변환
A_ub = [[-2, -1], [2, 1]]
b_ub = [-11, 95]
bounds = [(1, 2), (1, 75)]  # (P_electric, P_human)

### 5-1. Part A — 가이드북 그대로: 생산량과 무관한 정적 LP 1회 풀이

In [10]:
c_static = [1, 1]  # Cost = P_electric + P_human
res_static = linprog(c_static, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

P_electric_static, P_human_static = res_static.x
print(f"status: {'Optimal' if res_static.success else res_static.message}")
print(f"P_electric = {P_electric_static}")
print(f"P_human    = {P_human_static}")
print(f"Cost(최소) = {res_static.fun}")

status: Optimal
P_electric = 2.0
P_human    = 7.0
Cost(최소) = 9.0


**해석**: 가이드북이 공개한 3개 부등식만으로는 목적함수가 P_electric을 상한(2)까지, P_human을 하한 제약이 만족되는 최소값까지 밀어붙이는 '경계해'로 수렴한다.
가이드북 예시(공장직원 5.0명, 최종값 610.42)와 다른 값인데, 가이드북이 '생산량'을 이 LP에 연결하는 정확한 방식을 코드로 공개하지 않았기 때문이다
(목적함수 계수 또는 제약조건 우변이 생산량에 따라 매 시점 달라졌을 가능성이 높다).

### 5-2. Part B — 우리의 확장: 실제 인건비/전기요금(계절)을 비용계수로 모든 시간(행)에 반복 실행

※ 가이드북에 없는 우리의 해석 — 참고용

In [11]:
wages = df["인건비"].to_numpy()
elec_rates = df["전기요금(계절)"].to_numpy()
n = len(df)

P_electric_arr = np.empty(n)
P_human_arr = np.empty(n)
cost_arr = np.empty(n)

for i in range(n):
    res = linprog(
        [elec_rates[i], wages[i]],  # [P_electric 계수, P_human 계수]
        A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs",
    )
    P_electric_arr[i] = res.x[0]
    P_human_arr[i] = res.x[1]
    cost_arr[i] = res.fun

res_df = pd.DataFrame(
    {
        "index": df.index,
        "생산량": df["생산량"].to_numpy(),
        "인건비": wages,
        "전기요금(계절)": elec_rates,
        "추천_P_electric": P_electric_arr,
        "추천_P_human": P_human_arr,
        "최소비용": cost_arr,
    }
)
res_df.to_csv(RESULTS_DIR / "lp_optimization_result.csv", index=False)

total_cost = res_df["최소비용"].sum()
mean_human = res_df["추천_P_human"].mean()
mean_electric = res_df["추천_P_electric"].mean()

print(f"전체 {len(res_df)}개 시간(행)에 대한 LP 반복 실행 완료")
print(f"평균 추천 공장인원(P_human)     : {mean_human:.2f}")
print(f"평균 추천 전기설비가동(P_electric): {mean_electric:.2f}")
print(f"전체 시간 합산 최소비용 총계     : {total_cost:.4f}")

전체 6168개 시간(행)에 대한 LP 반복 실행 완료
평균 추천 공장인원(P_human)     : 9.00
평균 추천 전기설비가동(P_electric): 1.00
전체 시간 합산 최소비용 총계     : 1076826.9000


### 5-3. 참고 — 가이드북 예시 생산량(정규화 0.1718...)에 가장 가까운 실제 행

In [12]:
prod_min, prod_max = df["생산량"].min(), df["생산량"].max()
res_df["생산량_정규화"] = (res_df["생산량"] - prod_min) / (prod_max - prod_min)
closest = res_df.iloc[(res_df["생산량_정규화"] - 0.17183770899999998).abs().argsort()[:1]]
closest

,index,생산량,인건비,전기요금(계절),추천_P_electric,추천_P_human,최소비용,생산량_정규화
667,667,1690,1.5,109.8,1.0,9.0,123.3,0.171923


### 5-4. 결과 저장

In [13]:
with open(RESULTS_DIR / "lp_metrics.txt", "w") as f:
    f.write(f"static_cost={res_static.fun:.4f}\n")
    f.write(f"static_P_human={P_human_static}\n")
    f.write(f"static_P_electric={P_electric_static}\n")
    f.write(f"batch_total_cost={total_cost:.4f}\n")
    f.write(f"batch_mean_P_human={mean_human:.4f}\n")
    f.write(f"batch_mean_P_electric={mean_electric:.4f}\n")

print("[정보] lp_optimization_result.csv / lp_metrics.txt 저장 완료")

[정보] lp_optimization_result.csv / lp_metrics.txt 저장 완료
